# COSC2753 Assignment 2 — Fashion Intelligence System
# Task 4 — Chained Test-Prediction Pipeline

`styles_prediction.csv` contains 5,828 rows with an `id` and four empty columns. It carries
**no metadata at all** — no `baseColour`, no `year`, no `masterCategory`, no `subCategory`.
Every multi-input model in Tasks 1–3 was trained on those columns, so none of them can run
on the test set as-is.

This notebook closes that gap: predict the missing metadata from the images first, then feed
image + predicted metadata into the multi-input models.

**Part I/II below are the shared preprocessing pipeline** (identical to Tasks 1–3 — the
pipeline needs `train_data` for the label mappings and `val_data` to validate the chain).
**Part III is the pipeline itself.**


## I. Exploratory Data Analysis

### 1. Import Libraries

In [ ]:
import json
import pickle
import hashlib
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.stats import chi2_contingency

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold

import torch
from torch.utils.data import Dataset, WeightedRandomSampler
import torchvision.transforms as T

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
pd.set_option('display.max_columns', None)

### 2. Load Dataset

In [ ]:
DATA_DIR = Path("../data/raw/FashionDataset")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / "train" / "styles_train.csv"
IMAGES_TRAIN_DIR = DATA_DIR / "train" / "images_train"
TEST_PRED_CSV = DATA_DIR / "test" / "styles_prediction.csv"
IMAGES_TEST_DIR = DATA_DIR / "test" / "images_test"

for p in [TRAIN_CSV, IMAGES_TRAIN_DIR, TEST_PRED_CSV, IMAGES_TEST_DIR]:
    print(f"[{'OK' if p.exists() else 'MISSING'}] {p}")

## 3. Checking "style_train.csv" and images in the train dataset

#### 3.1 First look at the CSV

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print(df.columns.tolist())
print(df.shape)
df.head(3)

In [ ]:
# The raw CSV has stray commas in some rows, which pandas turns into extra
# 'Unnamed: N' columns. Drop them.
unnamed_cols = df.columns[df.columns.str.startswith('Unnamed')].tolist()
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
print(f"Dropped {len(unnamed_cols)} malformed column(s): {unnamed_cols}")
print(df.columns.tolist())
print(df.shape)

#### 3.2 Image check using csv file

Checking if any image file in "style_train.csv" doesn't match up with actual image

In [ ]:
csv_ids = set(df['id'].astype(str))
disk_ids = {p.stem for p in IMAGES_TRAIN_DIR.glob("*.jpg")}

missing_images = sorted(csv_ids - disk_ids)   # CSV rows with no image file
extra_images = sorted(disk_ids - csv_ids)     # image files with no CSV row

print(f"CSV rows with no image file: {len(missing_images)} -> {missing_images}")
print(f"Image files with no CSV row: {len(extra_images)}")

There are 5 csv row with no image file. These will be removed from the csv file.

In [ ]:
# Drop the unmatched rows now, before the split, so 'df' and its image folder
# stay aligned for every downstream step (EDA, split, processing).
df = df[~df['id'].astype(str).isin(missing_images)].reset_index(drop=True)
df['id'] = df['id'].astype(str).str.strip()
print(f"Remaining rows: {df.shape[0]}")

#### 3.3 Image integrity: corrupt files and exact duplicates

In [ ]:
# Corrupt / unopenable images
corrupt = []
for img_id in df['id']:
    try:
        with Image.open(IMAGES_TRAIN_DIR / f"{img_id}.jpg") as im:
            im.verify()
    except Exception as e:
        corrupt.append((img_id, str(e)))

print(f"Corrupt/unopenable images: {len(corrupt)}")
for img_id, err in corrupt[:10]:
    print(f"  {img_id}: {err}")

# Drop them if any turn up -- a no-op on the current data (0 corrupt), but it means
# the pipeline doesn't silently carry an unreadable file into the Dataset.
if corrupt:
    df = df[~df['id'].isin({i for i, _ in corrupt})].reset_index(drop=True)
    print(f"Dropped {len(corrupt)} corrupt row(s). Remaining: {len(df)}")

In [ ]:
# Exact duplicate images (byte-for-byte, via md5 hash). The resulting 'dup_group'
# is needed later: it's what the train/val split is grouped on, so two copies of
# the same product photo can never end up on opposite sides of the split.
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hash_to_group = {}
group_ids = []
for img_id in df['id']:
    h = file_hash(IMAGES_TRAIN_DIR / f"{img_id}.jpg")
    if h not in hash_to_group:
        hash_to_group[h] = img_id       # first id seen becomes the group label
    group_ids.append(hash_to_group[h])

df['dup_group'] = group_ids
n_groups = df['dup_group'].nunique()
n_dupe_rows = len(df) - n_groups
print(f"{n_groups} unique image groups out of {len(df)} rows "
      f"({n_dupe_rows} rows are exact duplicates of another row)")

#### 3.4 Quick visual sanity check

In [ ]:
sample = df.sample(12, random_state=RANDOM_STATE)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(IMAGES_TRAIN_DIR / f"{r['id']}.jpg"))
    ax.axis("off")
    ax.set_title(f"{r['articleType']}\n{r['baseColour']}", fontsize=7)
fig.suptitle("Random sample — sanity check that images load and labels look right", fontweight="bold")
plt.tight_layout()
plt.show()

### 4. Train / Validation Split

**Why not a plain random split:** a plain split doesn't check for duplicate images. We found 763 rows that are exact-duplicate copies of another row earlier — a plain random split could put one copy in train and the other in validation. That's leakage: the model would basically get tested on a picture it already trained on.

**Why we split on `masterCategory`, not `articleType`:** `articleType` has a lot of rare classes. To split evenly on it, we'd first need to decide which classes count as "rare" — and if we work that out using the full dataset (train and validation together), we'd be using validation labels to help set up training. `masterCategory` avoids this problem. It only has a few categories (Apparel, Accessories, Footwear, and so on) and all of them have plenty of samples, so we can split on it directly with no extra decisions needed. The `articleType` rare-class decision is made later, after the split, using only the train data (see Section 6.3).

**Trade-off:** splitting on `masterCategory` keeps the broad categories balanced between train and validation, but it doesn't guarantee every single `articleType` class is split perfectly evenly. With about 29,000 training rows, this is a small price to pay for avoiding leakage completely.

**What we do:** one `StratifiedGroupKFold` split (about 75/25) — stratified on `masterCategory`, grouped on `dup_group` so duplicate images always stay on the same side. We only use one split, not full cross-validation: with ~29,000 rows this is enough data for a stable result, and doing 4-fold cross-validation would mean training every model four times over — time better spent improving the models themselves.

#### 4.1 Doing the split

One more thing before splitting: `masterCategory` needs at least a handful of rows in every one of its categories for the split to work properly. A category with only one or two rows total can't be divided across train and validation at all. Since `masterCategory` isn't a prediction target for any task, dropping a tiny number of rows here doesn't touch anything the model is being trained to predict — it's a mechanical fix, not a modeling decision.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)  # 1 fold ~= 25%

# masterCategory needs at least n_splits rows per category for a clean split. A category
# with fewer rows than that can't be divided properly and ends up placed by chance --
# drop those rows first rather than let that happen silently.
master_counts = df['masterCategory'].value_counts()
too_rare_master = master_counts[master_counts < sgkf.get_n_splits()].index
if len(too_rare_master):
    n_dropped = df['masterCategory'].isin(too_rare_master).sum()
    print(f"Dropping {n_dropped} row(s) with a masterCategory that has fewer than "
          f"{sgkf.get_n_splits()} samples total: {list(too_rare_master)}")
    df = df[~df['masterCategory'].isin(too_rare_master)].reset_index(drop=True)

train_idx, val_idx = next(sgkf.split(df, df['masterCategory'], groups=df['dup_group']))
train_data = df.iloc[train_idx].reset_index(drop=True)
val_data = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {train_data.shape[0]}, Val: {val_data.shape[0]}")

# --- sanity checks ---
overlap = set(train_data['dup_group']) & set(val_data['dup_group'])
print(f"Duplicate-image groups appearing in BOTH splits: {len(overlap)} (should be 0)")

missing_master = set(df['masterCategory'].unique()) - set(train_data['masterCategory'].unique())
print(f"masterCategory classes missing from train: {missing_master if missing_master else 'none'}")

### 5. Exploring the Train Data

This only looks at `train_data`, after the split — as the course asks, and so nothing below is influenced by the validation data.

#### 5.1 Structure & overview

In [ ]:
print(train_data.columns.tolist())
print(train_data.shape)
train_data.head()

In [ ]:
train_data.info()

#### 5.2 Missing values check

In [ ]:
missing = train_data.isnull().sum()
missing_pct = (missing / len(train_data)) * 100
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct.round(2)}).sort_values('missing', ascending=False)
missing_df

In [ ]:
nonzero_missing = missing[missing > 0].sort_values(ascending=False)
if len(nonzero_missing):
    plt.figure(figsize=(6, 4))
    sns.barplot(x=nonzero_missing.values, y=nonzero_missing.index)
    plt.title('Missing Values by Column (train split)')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.show()

#### 5.3 Duplicate rows in metadata

In [ ]:
print(f"Duplicate rows: {train_data.duplicated().sum()}")
print(f"Duplicate ids: {train_data['id'].duplicated().sum()}")

#### 5.4 Category distributions

In [ ]:
cat_cols = ['gender', 'masterCategory', 'subCategory', 'baseColour', 'season', 'usage', 'articleType', 'year']
fig, axes = plt.subplots(4, 2, figsize=(14, 18))  # 4x2 = 8 axes, one per cat_col
for ax, col in zip(axes.flatten(), cat_cols):
    order = train_data[col].value_counts().index[:15]  # top 15 to keep it readable
    sns.countplot(data=train_data, y=col, order=order, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
print("Category distributions (summary):\n")
for col in cat_cols:
    vc = train_data[col].value_counts()
    top_val, top_count = vc.index[0], vc.iloc[0]
    top_pct = top_count / len(train_data) * 100
    n_classes = vc.shape[0]
    least_val, least_count = vc.index[-1], vc.iloc[-1]
    print(f"- {col}: {n_classes} categories. "
          f"Most common is '{top_val}' ({top_count} items, {top_pct:.1f}%). "
          f"Least common is '{least_val}' ({least_count} items).")

#### 5.5 Closer check at "articleType"

In [ ]:
vc = train_data['articleType'].value_counts()
print(f"Distinct articleType classes in train: {train_data['articleType'].nunique()}")
print(f"Classes with fewer than 10 samples: {(vc < 10).sum()}")
print(f"Classes with fewer than 5 samples:  {(vc < 5).sum()}")
vc.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(vc) + 1), vc.values, marker='o', markersize=3)
ax.set_yscale('log')
ax.set_xlabel('Class rank')
ax.set_ylabel('Count (log scale)')
ax.set_title(f'articleType long-tail distribution ({len(vc)} classes)')
plt.tight_layout()
plt.show()

In [ ]:
N = 25
plt.figure(figsize=(8, 10))
top_article = train_data['articleType'].value_counts().index[:N]
sns.countplot(data=train_data, y='articleType', order=top_article)
plt.title(f'Top {N} Article Types')
plt.tight_layout()
plt.show()

#### 5.6 Year distribution & outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train_data['year'], bins=train_data['year'].nunique(), ax=axes[0])
axes[0].set_title('Year Distribution')
sns.boxplot(x=train_data['year'], ax=axes[1])
axes[1].set_title('Year Outliers')
plt.tight_layout()
plt.show()

In [ ]:
year_counts = train_data['year'].value_counts().sort_index()

q1 = train_data['year'].quantile(0.25)
q3 = train_data['year'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_years = train_data[(train_data['year'] < lower_bound) | (train_data['year'] > upper_bound)]['year']
outlier_summary = outlier_years.value_counts().sort_index()

print(f"Data spans {train_data['year'].min():.0f} to {train_data['year'].max():.0f}.")
print(f"Most items are from {year_counts.idxmax():.0f} ({year_counts.max()} items).")
print(f"Outlier years (IQR rule): {outlier_summary.to_dict()}")

#### 5.7 Cramér's V — how strongly the feature columns are related

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    return np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

# 'productDisplayName' is deliberately excluded: it is near-unique free text (28,954
# distinct values), so its crosstab is enormous and Cramer's V against it is ~1 by
# construction -- it measures uniqueness, not a real association.
feature_cols = ['gender', 'masterCategory', 'subCategory', 'season',
                'usage', 'baseColour', 'year', 'articleType']
corr_matrix = pd.DataFrame(index=feature_cols, columns=feature_cols, dtype=float)
for c1, c2 in combinations(feature_cols, 2):
    v = cramers_v(train_data[c1], train_data[c2])
    corr_matrix.loc[c1, c2] = v
    corr_matrix.loc[c2, c1] = v
for c in feature_cols:
    corr_matrix.loc[c, c] = 1.0

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix.astype(float), annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.title("Cramér's V — relations between feature columns")
plt.tight_layout()
plt.show()

#### 5.8 Target distributions & relationships between targets

In [ ]:
targets = ['articleType', 'gender', 'season', 'usage']

# All four targets are already in `corr_matrix` from 5.7 -- slice it rather than
# recomputing the same chi-square tests.
target_corr = corr_matrix.loc[targets, targets].astype(float)

plt.figure(figsize=(6, 5))
sns.heatmap(target_corr, annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.title("Cramer's V - relations between the four prediction targets")
plt.tight_layout()
plt.show()


In [ ]:
# The strongest target-target relationship, viewed as a crosstab heatmap
strongest_pair = target_corr.where(~np.eye(len(targets), dtype=bool)).stack().idxmax()
c1, c2 = strongest_pair
print(f"Strongest target relationship: {c1} vs {c2} (Cramér's V = {target_corr.loc[c1, c2]:.2f})")

plt.figure(figsize=(8, 10))
ct = pd.crosstab(train_data[c1], train_data[c2])
top_types = train_data[c1].value_counts().index[:25]
sns.heatmap(ct.loc[ct.index.intersection(top_types)], annot=True, fmt='d', cmap='Blues')
plt.title(f'{c1} vs {c2} (top 25 {c1} classes)')
plt.tight_layout()
plt.show()

#### 5.9 How imbalanced are the classes

In [ ]:
def imbalance_summary(series, name):
    vc = series.dropna().value_counts()
    ratio = vc.max() / vc.min()
    return {
        'target': name,
        'n_classes': len(vc),
        'majority_class': vc.idxmax(),
        'majority_count': int(vc.max()),
        'minority_class': vc.idxmin(),
        'minority_count': int(vc.min()),
        'imbalance_ratio': round(ratio, 1),
    }

imbalance_table = pd.DataFrame([
    imbalance_summary(train_data['gender'], 'gender'),
    imbalance_summary(train_data['season'], 'season'),
    imbalance_summary(train_data['usage'], 'usage'),
    imbalance_summary(train_data['articleType'], 'articleType'),
])
imbalance_table

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), ['gender', 'season', 'usage', 'articleType']):
    shares = train_data[col].value_counts(normalize=True, dropna=False) * 100
    if col == 'articleType':
        shares = shares.head(25)
    shares.plot(kind='bar', ax=ax, color='#C44E52')
    ax.set_ylabel('% of rows')
    ax.set_title(f'{col} — class share (%)')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

#### 5.10 Checking the images: size, color mode, file size (train data only)

In [ ]:
def profile_images(ids, folder):
    rows = []
    for img_id in ids:
        p = folder / f"{img_id}.jpg"
        try:
            with Image.open(p) as im:      # reads header only, fast
                w, h, mode = im.size[0], im.size[1], im.mode
        except Exception:
            w = h = None
            mode = "ERR"
        rows.append({"id": img_id, "w": w, "h": h, "mode": mode,
                     "kb": round(p.stat().st_size / 1024, 2)})
    return pd.DataFrame(rows)

imgs_train = profile_images(train_data['id'], IMAGES_TRAIN_DIR)
print("Most common dimensions (train):")
print(imgs_train.groupby(['w', 'h']).size().sort_values(ascending=False).head())
print("\nColour mode counts (train):", imgs_train['mode'].value_counts().to_dict())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(imgs_train["kb"], bins=60, color="#4C72B0")
ax[0].set(title="Image file size distribution (train)", xlabel="KB", ylabel="count")

mode_counts = imgs_train["mode"].value_counts()
ax[1].bar(mode_counts.index.astype(str), mode_counts.values, color="#55A868")
ax[1].set_yscale("log")
ax[1].set(title="Colour mode (log scale)", ylabel="count")
plt.tight_layout()
plt.show()

**What we found (train split only, 28,958 rows):**
- 28,958 training images, almost all a consistent 60x80 pixels, with 12 size outliers.
- 249 images are grayscale (`L` mode), not colour -- these need converting to RGB before
  going into a CNN (done in the transform pipeline, Section II.8).
- File sizes are mostly small (under 25 KB), and no image failed the integrity check.
- `articleType` has a long tail -- 39 classes with fewer than 10 samples. Addressed in Section II.3.
- `usage` is 76.8% `Casual`, and `gender` is mostly `Men`/`Women` -- both handled by the
  imbalance fix in Section II.6.
- None of the feature and target columns are strongly linked enough to drop any of them.


## II. Preprocessing data

This section builds the four datasets (`articleType`, `season`, `gender`, `usage`) that Tasks 1-3 will train on, and explains the reasoning behind each decision.

### 1 Basic cleaning

This same cleaning function runs on both `train_data` and `val_data`. It doesn't learn anything from the data (no fitting), so using the same function on both is safe — no leakage risk.

In [ ]:
def clean_dataframe(data):
    """Whitespace-strip categorical/text columns, coerce year to numeric,
    and fill non-target missing values with a placeholder. Fit-free — safe
    to reuse on train, val, or a future test split."""
    data = data.copy()

    # Note: 'articleType_grouped' isn't in this list -- it doesn't exist yet at this point
    # in the pipeline (created in Section 6.3, after cleaning), and it derives from
    # already-cleaned 'articleType'/'subCategory' values, so it needs no separate stripping.
    cat_cols = ['gender', 'masterCategory', 'subCategory', 'articleType',
                'baseColour', 'season', 'usage', 'productDisplayName']
    for c in cat_cols:
        if c in data.columns:
            data[c] = data[c].astype('string').str.strip()

    if 'year' in data.columns:
        data['year'] = pd.to_numeric(data['year'], errors='coerce')

    # baseColour / productDisplayName aren't prediction targets, so missing values
    # here don't block any task — fill rather than drop.
    for c in ['baseColour', 'productDisplayName']:
        if c in data.columns:
            data[c] = data[c].fillna('Unknown')

    return data

train_data = clean_dataframe(train_data)
val_data = clean_dataframe(val_data)

print("Missing values after cleaning (train):")
print(train_data.isnull().sum())
print("\nMissing values after cleaning (val):")
print(val_data.isnull().sum())

### 2. What to do about missing values, per task

`season` and `usage` are prediction targets, so we can't fill in a missing value without just guessing a label. Rows missing those values get dropped, but only for that one task — not from the shared `train_data`/`val_data`. `articleType` and `gender` have no missing values. This is handled in Section 6.4, where we build a separate dataset for each task.

### 3. Handling rare classes, per task

**`articleType`** — this is worked out using train data only, so validation rows are never looked at. Classes with too few samples get relabelled to their own `subCategory` instead of a single "Other" bucket. For example, a rare class like "Rain Trousers" becomes "Bottomwear". This keeps more useful information than dumping everything into one catch-all label. One trade-off worth mentioning in the report: this mixes two levels of detail in one column — some rows keep a specific label like "T-shirts", others become a broader one like "Bottomwear" after being redistributed. A validation row whose `articleType` never appears in train at all (so it can't be judged "rare" from train counts, since it has zero train count, not just a low one) gets caught and dropped automatically later, at the label-encoding step in Section II.5 — not handled here, so this step never has to look at validation data.

In [ ]:
# (The class counts themselves were shown in Section 5.5 -- this cell only sweeps
# candidate thresholds to justify the value picked below.)
vc_article = train_data['articleType'].value_counts()

for thresh in [5, 10, 15, 20, 30, 50]:
    kept_classes = (vc_article >= thresh).sum()
    kept_rows = vc_article[vc_article >= thresh].sum()
    print(f"threshold={thresh:>3}: classes kept={kept_classes:>3}/{len(vc_article)}, "
          f"rows kept={kept_rows:>6} ({kept_rows/len(train_data)*100:.1f}% of train)")


In [ ]:
ARTICLE_TYPE_MIN_COUNT = 20  # keeps most classes while dropping the near-unlearnable long tail

rare_article_types = set(vc_article[vc_article < ARTICLE_TYPE_MIN_COUNT].index)

def apply_article_type_grouping(data, rare_set):
    """Redistributes rare articleType rows into their own subCategory, rather than a
    single 'Other' bucket. `rare_set` is fixed from train-only counts and applied as-is
    to both splits -- never recomputed from val."""
    data = data.copy()
    rare_mask = data['articleType'].isin(rare_set)
    data['articleType_grouped'] = data['articleType']
    data.loc[rare_mask, 'articleType_grouped'] = data.loc[rare_mask, 'subCategory']
    return data

train_data = apply_article_type_grouping(train_data, rare_article_types)
val_data = apply_article_type_grouping(val_data, rare_article_types)

print(f"Rare articleType classes folded into subCategory: {len(rare_article_types)}")
print("Classes after grouping (train):", train_data['articleType_grouped'].nunique())
print("\nRare rows redistributed into (train):")
print(train_data.loc[train_data['articleType'].isin(rare_article_types), 'articleType_grouped'].value_counts())

In [ ]:
# Verify the redistribution actually fixed the imbalance -- if a subCategory a rare
# articleType got folded into is itself still small, the long tail has moved, not gone.
post_counts = train_data['articleType_grouped'].value_counts()
still_rare = post_counts[post_counts < ARTICLE_TYPE_MIN_COUNT]
print(f"articleType_grouped classes still below {ARTICLE_TYPE_MIN_COUNT} samples after redistribution: {len(still_rare)}")
if len(still_rare):
    print(still_rare)
    print("\nNOTE: the long tail has been reduced but not eliminated -- the smallest classes\n"
          "here still have single-digit support. Report macro-F1 per class so this is visible,\n"
          "and treat these classes' scores as unreliable rather than as model failure.")

**`usage`** doesn't have a parent column to redistribute into, so rare classes here go
into a single `Other` bucket. The three classes folded in account for 66 rows between
them, so `Other` stays small enough not to distort the remaining four classes. Same
train-only approach as `articleType`:


In [ ]:
USAGE_MIN_COUNT = 100

vc_usage = train_data['usage'].value_counts()
for thresh in [10, 25, 50, 100, 150]:
    kept_classes = (vc_usage >= thresh).sum()
    kept_rows = vc_usage[vc_usage >= thresh].sum()
    print(f"threshold={thresh:>3}: classes kept={kept_classes}/{len(vc_usage)}, "
          f"rows kept={kept_rows} ({kept_rows/train_data['usage'].notna().sum()*100:.1f}% of usage rows)")

In [ ]:
rare_usage = vc_usage[vc_usage < USAGE_MIN_COUNT].index

for d in (train_data, val_data):
    d['usage_grouped'] = d['usage'].where(~d['usage'].isin(rare_usage), 'Other')

print("usage classes after grouping (train):", train_data['usage_grouped'].nunique())
print(train_data['usage_grouped'].value_counts())

**`gender`** and **`season`** don't need any merging. `gender` has 5 classes and `season` has 4. Both are imbalanced (see Section 5.9), but every class still has enough samples to split, encode, and learn from without needing to combine categories.

### 4. One dataset per task

In [ ]:
# Column actually used as the prediction target for each task
target_columns = {
    'articleType': 'articleType_grouped',
    'season': 'season',
    'gender': 'gender',
    'usage': 'usage_grouped',
}

def usable_subset(data, target_col):
    return data[data[target_col].notna()].reset_index(drop=True)

train_usable = {t: usable_subset(train_data, col) for t, col in target_columns.items()}
val_usable = {t: usable_subset(val_data, col) for t, col in target_columns.items()}

for t in target_columns:
    print(f"{t}: train usable={len(train_usable[t])}, val usable={len(val_usable[t])}")

#### 4.1 A separate split for Task 2 (`season`)

`season` isn't correlated with `masterCategory` the way `articleType` is — a "Summer" item can be a shirt, a shoe, or a bag, cutting across every `masterCategory` roughly evenly. So the shared `masterCategory`-stratified split above gives no real guarantee that `season` classes are proportionally represented between train and validation for Task 2. Here we redo the split for Task 2 only, stratified directly on `season`, keeping the same duplicate-image grouping constraint so leakage is still prevented. This overwrites `train_usable['season']`/`val_usable['season']` before anything downstream uses them, so Tasks 1, 3, and 4 are unaffected.

One caveat to note in the report: this split is drawn from `df` (the full dataset), so a row in the *main* train split may land in the *season* validation split. That is fine for Task 2 in isolation, since Task 2's own train and validation sides stay disjoint and duplicate-grouped, but it does mean the normalisation statistics in Section II.7 (computed over main-train images) have seen a minority of season-validation images. The effect on a channel mean/std over 3,000 images is negligible, but it is a real, acknowledged approximation rather than a clean separation.

In [ ]:
def class_balance(train_df, val_df, col):
    """Train% vs val% per class, sorted by the biggest gap -- used to sanity-check
    how representative a split is for a given target column."""
    comp = pd.DataFrame({
        'train_%': train_df[col].value_counts(normalize=True).sort_index(),
        'val_%':   val_df[col].value_counts(normalize=True).sort_index(),
    }).round(4)
    comp['abs_diff'] = (comp['train_%'] - comp['val_%']).abs()
    return comp.sort_values('abs_diff', ascending=False)

print('Season balance -- shared masterCategory-stratified split (before):')
print(class_balance(train_usable['season'], val_usable['season'], 'season'))

sgkf_season = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

# Only rows with a known season can be stratified on it -- same 'drop, don't impute'
# decision as for the target itself elsewhere in this notebook.
# clean_dataframe() must be applied here too: `df` is the *raw* frame, so without this
# the season labels are unstripped and would not match the cleaned values used everywhere
# else (e.g. 'Summer ' and 'Summer' would encode as two separate classes).
df_season_pool = clean_dataframe(df)
df_season_pool = df_season_pool[df_season_pool['season'].notna()].reset_index(drop=True)

season_counts = df_season_pool['season'].value_counts()
too_rare_season = season_counts[season_counts < sgkf_season.get_n_splits()].index
if len(too_rare_season):
    n_dropped = df_season_pool['season'].isin(too_rare_season).sum()
    print(f"Dropping {n_dropped} row(s) with a season that has fewer than "
          f"{sgkf_season.get_n_splits()} samples total: {list(too_rare_season)}")
    df_season_pool = df_season_pool[~df_season_pool['season'].isin(too_rare_season)].reset_index(drop=True)

season_train_idx, season_val_idx = next(
    sgkf_season.split(df_season_pool, df_season_pool['season'], groups=df_season_pool['dup_group'])
)
season_train_data = df_season_pool.iloc[season_train_idx].reset_index(drop=True)
season_val_data = df_season_pool.iloc[season_val_idx].reset_index(drop=True)

overlap = set(season_train_data['dup_group']) & set(season_val_data['dup_group'])
print(f"\nTask 2 split -- duplicate-image groups appearing in BOTH sides: {len(overlap)} (should be 0)")
print(f"Task 2 split -- train: {len(season_train_data)}, val: {len(season_val_data)}")

# Overwrite train_usable/val_usable['season'] so every downstream cell (label encoding,
# samplers, datasets, config export, leakage checks) automatically uses this split for
# Task 2 -- no other code changes needed anywhere else in the notebook.
train_usable['season'] = usable_subset(season_train_data, 'season')
val_usable['season'] = usable_subset(season_val_data, 'season')

print('\nSeason balance -- season-stratified split (after):')
print(class_balance(train_usable['season'], val_usable['season'], 'season'))

print(f"\nseason: train usable={len(train_usable['season'])}, val usable={len(val_usable['season'])}")


### 5. Turning labels into numbers

Each task gets its own label encoder, fit on train data only — since each task drops different rows, one shared encoder wouldn't make sense across all of them. If validation ever has a label the encoder never saw in train (shouldn't happen after our split, but we check anyway), that row gets flagged and dropped instead of silently causing an error.

In [ ]:
encoders = {}
label_maps = {}

for t, col in target_columns.items():
    le = LabelEncoder()
    le.fit(train_usable[t][col])
    encoders[t] = le
    label_maps[t] = {i: c for i, c in enumerate(le.classes_)}

    train_usable[t][col + '_enc'] = le.transform(train_usable[t][col])

    val_known = val_usable[t][val_usable[t][col].isin(le.classes_)]
    dropped = len(val_usable[t]) - len(val_known)
    if dropped:
        print(f"WARNING: dropped {dropped} val rows for '{t}' — label(s) not seen in train")
    val_usable[t] = val_known.reset_index(drop=True)
    val_usable[t][col + '_enc'] = le.transform(val_usable[t][col])

    print(f"{t}: {len(le.classes_)} classes" + (f" -> {label_maps[t]}" if len(le.classes_) <= 6 else ""))

### 6. Fixing class imbalance

**Options we thought about:** giving rare classes more weight in the loss function, oversampling them, or augmenting their images. Using all three together tends to over-correct, so we picked one consistent approach for all four tasks instead of tuning something different for each.

**What we're using: `WeightedRandomSampler`**, together with the augmentation from Section 6.8. This is built from training rows only — validation is never resampled, so it still reflects the real, imbalanced distribution.

- **Why not also weight the loss function:** once the sampler is already balancing each batch, adding loss weighting on top would double-punish the common classes — risky for `articleType`, where some of those weights would be very small. We calculate class weights below just for reference, but don't actually use them in training.
- **Why the sampler needs augmentation:** the sampler picks the same rare-class images again and again (with replacement). Without variation, the model could just memorize those exact images instead of learning to generalize. Augmentation (flip, rotate, adjust color — Section 6.8) makes each repeat draw look a bit different.
- **Why not SMOTE:** SMOTE blends pixels between different photos to make synthetic examples, which for product photos just produces unrealistic-looking images. Augmentation does the same job in a way that makes sense for images.
- **One more thing to track:** always look at macro-F1 next to accuracy in the modelling notebooks. Accuracy alone can be misleading here — a `usage` model that only ever predicts "Casual" would still score around 77% accuracy while being useless.

In [ ]:
# Reference-only class weights (not used in the training loss)
class_weights = {}
for t, col in target_columns.items():
    data = train_usable[t]
    vc = data[col].value_counts().sort_index()
    n_classes, n_samples = len(vc), len(data)
    weights = n_samples / (n_classes * vc)
    weights = weights / weights.mean()
    ordered = [weights[c] for c in encoders[t].classes_]
    class_weights[t] = np.array(ordered)
    print(f"{t}: weight range {class_weights[t].min():.2f}-{class_weights[t].max():.2f}")

In [ ]:
def get_weighted_sampler(data, target_col):
    """WeightedRandomSampler that oversamples minority classes within `data`.
    Call on the training subset only."""
    class_counts = data[target_col].value_counts()
    inv_freq = {cls: 1.0 / count for cls, count in class_counts.items()}
    sample_weights = torch.tensor(data[target_col].map(inv_freq).values, dtype=torch.float32)
    return WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

samplers = {}
for t, col in target_columns.items():
    samplers[t] = get_weighted_sampler(train_usable[t], col)
    print(f"{t}: sampler built on {len(train_usable[t])} training rows")

### 7. Image size and normalization

**Size:** we keep the images at their natural 60x80 size (width x height) instead of forcing them into a square. Squaring them would stretch every image out of shape for no real benefit, since 60x80 is already the size almost all images already are (see Section 5.10).

**Normalization:** we calculate the average and spread of pixel values (per color channel) from a sample of train images, instead of just dividing by 255. This is standard practice for training CNNs — it centers the pixel values around zero, which a plain 0-to-1 scale doesn't do.

In [ ]:
IMG_WIDTH, IMG_HEIGHT = 60, 80

sample_ids = train_data['id'].sample(min(3000, len(train_data)), random_state=RANDOM_STATE)

pixel_sum = np.zeros(3)
pixel_sq_sum = np.zeros(3)
n_pixels = 0

for img_id in sample_ids:
    img = Image.open(IMAGES_TRAIN_DIR / f"{img_id}.jpg").convert("RGB")
    arr = np.asarray(img, dtype=np.float64) / 255.0
    pixel_sum += arr.sum(axis=(0, 1))
    pixel_sq_sum += (arr ** 2).sum(axis=(0, 1))
    n_pixels += arr.shape[0] * arr.shape[1]

mean = pixel_sum / n_pixels
std = np.sqrt(pixel_sq_sum / n_pixels - mean ** 2)

print(f"Computed mean (RGB): {mean.round(4)}")
print(f"Computed std (RGB):  {std.round(4)}")

### 8. Transform pipelines

In [ ]:
def to_rgb(img):
    """Handles the 249 grayscale ('L' mode) files found in Section 5.10. A module-level
    function rather than a lambda, so the transform stays picklable -- a lambda breaks
    DataLoader(num_workers>0) on any spawn-based platform (Windows/macOS)."""
    return img.convert("RGB")


# Evaluation pipeline -- used for validation, test, and as the basis of the training
# pipeline. Never augmented, so it stays a faithful evaluation signal.
eval_transform = T.Compose([
    to_rgb,
    T.Resize((IMG_HEIGHT, IMG_WIDTH)),
    T.ToTensor(),
    T.Normalize(mean=mean.tolist(), std=std.tolist()),
])

# Training pipeline -- adds mild, conservative augmentation. Kept conservative because
# these are catalog product photos, not natural scenes: a vertical flip or a large
# rotation would produce an unrealistic example (e.g. an upside-down shoe).
# fill=255 matters: these photos sit on a white background (channel means ~0.85), so the
# default fill=0 would paste black wedges into the corners of every rotated image and
# hand the model an artefact that never appears at evaluation time.
train_transform = T.Compose([
    to_rgb,
    T.Resize((IMG_HEIGHT, IMG_WIDTH)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10, fill=255),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=mean.tolist(), std=std.tolist()),
])

print("Transforms ready: eval_transform (no augmentation), train_transform (augmented)")


Augmentation isn't targeted at rare classes specifically — it just naturally works well with the sampler from Section II.6. Since rare-class images get picked more often by the sampler, they also get augmented more often, which is exactly the extra variety they need.

### 9. Wrapping everything in a PyTorch Dataset

In [ ]:
class FashionImageDataset(Dataset):
    """Wraps a per-task dataframe (from train_usable / val_usable) with its images
    and label encoder. Pass `train_transform` for training data, `eval_transform`
    for validation/test data."""

    def __init__(self, dataframe, images_dir, target_col_enc, transform):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = images_dir
        self.target_col_enc = target_col_enc
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(self.images_dir / f"{row['id']}.jpg") as im:
            img = self.transform(im)
        label = int(row[self.target_col_enc])
        return img, torch.tensor(label, dtype=torch.long)


# Example: build the four training datasets (validation datasets follow the same pattern
# with eval_transform and no sampler)
task_datasets = {
    t: FashionImageDataset(train_usable[t], IMAGES_TRAIN_DIR, col + '_enc', train_transform)
    for t, col in target_columns.items()
}
for t, ds in task_datasets.items():
    print(f"{t}: {len(ds)} training images")

### 10. Saving all our settings in one file

Every threshold, seed, and decision made in this notebook gets saved here in one place, so a teammate or a marker can see what was decided without having to re-run everything above.

In [ ]:
config = {
    "random_state": RANDOM_STATE,
    "split": {
        "method": "single_stratified_group_holdout",
        "splitter": "StratifiedGroupKFold(n_splits=4), first split only",
        "validation_fraction": 0.25,
        "stratify_on": {
            "articleType": "masterCategory",  # deliberately not articleType itself -- see Section 4
            "gender": "masterCategory",
            "usage": "masterCategory",
            "season": "season",  # re-split for Task 2 only -- see Section II.4.1
        },
        "group_on": "dup_group (exact-duplicate images)",
    },
    "image": {
        "width": IMG_WIDTH,
        "height": IMG_HEIGHT,
        "normalization_mean": mean.tolist(),
        "normalization_std": std.tolist(),
    },
    "rare_class_thresholds": {
        "articleType": ARTICLE_TYPE_MIN_COUNT,
        "usage": USAGE_MIN_COUNT,
    },
    "rare_class_handling": {
        "articleType": "redistributed into subCategory (train-only counts + val-only-class check)",
        "usage": "redistributed into 'Other' (no parent column to redistribute into)",
    },
    "class_counts": {t: int(len(encoders[t].classes_)) for t in target_columns},
    "imbalance_handling": "WeightedRandomSampler (train only) + augmentation; no loss-level class weighting",
    "missing_images_dropped": missing_images,
    "duplicate_image_rows_found": len(df) - df['dup_group'].nunique(),  # rows, not pairs
    "eda_scope": "train_data only, post-split, per course instruction",
}

with open(OUT_DIR / 'pipeline_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))

### 11. Saving the processed files

In [ ]:
export_dir = OUT_DIR / 'holdout_metadata'
export_dir.mkdir(parents=True, exist_ok=True)

for t in target_columns:
    train_usable[t].to_csv(export_dir / f'{t}_train.csv', index=False)
    val_usable[t].to_csv(export_dir / f'{t}_val.csv', index=False)
    print(f"{t}: exported {len(train_usable[t])} train and {len(val_usable[t])} validation rows")

with open(OUT_DIR / 'label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

mapping_dir = export_dir / 'label_mappings'
mapping_dir.mkdir(exist_ok=True)
for t, encoder in encoders.items():
    (mapping_dir / f'{t}.json').write_text(
        json.dumps({str(i): label for i, label in enumerate(encoder.classes_)}, indent=2),
        encoding='utf-8',
    )

# Full cleaned train/val (all columns, all rows) kept for reference ONLY.
# These reflect the MAIN masterCategory-stratified split. Task 2 (season) uses the
# separate split from Section II.4.1 -- always load season_train.csv / season_val.csv
# for that task, never these two files.
train_data.to_csv(OUT_DIR / 'train_full.csv', index=False)
val_data.to_csv(OUT_DIR / 'val_full.csv', index=False)

print("\nSaved to:", OUT_DIR.resolve())

### 12. Final checks: no leakage, and the saved files are correct

In [ ]:
# Leakage / integrity checks
# Checked against files on disk (not train_data ids) -- val ids are never a subset
# of train ids by design, since the split makes them disjoint on purpose.
train_image_ids = {p.stem for p in IMAGES_TRAIN_DIR.glob('*.jpg')}

for t, col in target_columns.items():
    tr, va = train_usable[t], val_usable[t]
    assert set(tr['id']).isdisjoint(set(va['id'])), f"{t}: id overlap between train/val"
    assert set(tr['dup_group']).isdisjoint(set(va['dup_group'])), f"{t}: duplicate-image group overlap"
    assert tr[col].notna().all() and va[col].notna().all(), f"{t}: unexpected missing target"
    assert set(tr['id']).issubset(train_image_ids), f"{t}: train id(s) with no matching image file"
    assert set(va['id']).issubset(train_image_ids), f"{t}: val id(s) with no matching image file"

print("All leakage/integrity checks passed: no id or duplicate-image-group overlap between splits, "
      "no missing targets, all ids resolve to real images.")

In [ ]:
# Reload sanity check — confirms what's on disk actually matches what's in memory
for t in target_columns:
    reloaded_train = pd.read_csv(export_dir / f'{t}_train.csv')
    reloaded_val = pd.read_csv(export_dir / f'{t}_val.csv')

    assert len(reloaded_train) == len(train_usable[t]), f"train row count mismatch for {t}"
    assert len(reloaded_val) == len(val_usable[t]), f"val row count mismatch for {t}"

    print(f"{t}: OK — train={len(reloaded_train)}, val={len(reloaded_val)}, "
          f"classes={reloaded_train[target_columns[t]].nunique()}")

with open(OUT_DIR / 'label_encoders.pkl', 'rb') as f:
    reloaded_encoders = pickle.load(f)
print("Encoders reload OK:", list(reloaded_encoders.keys()))

## III. The Chained Pipeline

### The dependency problem, and how the chain avoids it

Each task's multi-input model needs metadata that the test set doesn't have:

| Model | Metadata it requires |
|---|---|
| Task 1 (`articleType`) | `gender`, `baseColour`, `season`, `usage`, `year` |
| Task 2 (`season`) | `gender`, `masterCategory`, `subCategory`, `articleType`, `baseColour`, `year`, `usage` |
| Task 3 (`gender`, `usage`) | `masterCategory`, `subCategory`, `articleType`, `baseColour`, `year`, `season` |

The union of what must be generated is eight columns. Note the circularity: Task 1 needs
`gender`, which is Task 3's target; Task 3 needs `articleType`, which is Task 1's target;
Task 2 needs both. **A multi-input model cannot supply another multi-input model's inputs** —
that's a cycle with no starting point.

The chain breaks it with two strictly ordered stages:

- **Stage 1 — attribute generation, image-only models only.** Every metadata column is
  produced from pixels alone. Nothing in this stage consumes metadata, so nothing is circular.
- **Stage 2 — final prediction, multi-input models.** These consume Stage 1's output. Nothing
  in Stage 2 feeds back into Stage 1.

Where each of the eight columns comes from:

| Column | Source | Notes |
|---|---|---|
| `articleType` | Task 1 best image-only CNN | from-scratch |
| `gender` | Task 3 image-only CNN | from-scratch |
| `usage` | Task 3 image-only CNN | from-scratch |
| `season` | Task 2 best image-only CNN | from-scratch |
| `baseColour` | **new CNN, trained in this notebook** | no task predicts it |
| `subCategory` | derived from predicted `articleType` | deterministic lookup, verified below |
| `masterCategory` | derived from predicted `subCategory` | deterministic lookup, verified below |
| `year` | mode of the training set | constant; no model |

### The honest caveat, stated up front

The multi-input models were trained on **true** metadata and will be served **predicted**
metadata. That is train/serve skew: a model that learned to trust a clean `gender` signal
will be fed a noisy one, and Task 1's own ablation (Step 11f) found `gender` was the single
largest contributor to its 0.750. So the chain **may well score worse than simply using the
image-only models**.

That is an empirical question, not a rhetorical one, so Section 7 measures it directly: the
whole chain is re-run on the validation split using predicted metadata, and compared against
both the oracle-metadata score and the image-only score. Section 9 then picks whichever
actually wins, per task. Building the chain and discovering it doesn't help is a real result
worth reporting — it is not a reason to skip the measurement.

### 1. Configuration and artifact paths

Every model this notebook loads was trained and saved by Tasks 1–3. Run those three
notebooks first; this one trains only the `baseColour` model, which no task produces.

In [ ]:
import joblib
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, NUM_WORKERS = 128, 4
print("Device:", DEVICE)

TASK1_DIR = Path("../outputs/task1_models")
TASK2_DIR = Path("../outputs/task2_models")
TASK3_DIR = Path("../outputs/task3_models")
PIPELINE_DIR = Path("../outputs/pipeline")
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

# Fail early and specifically rather than deep inside a load call.
REQUIRED_ARTIFACTS = {
    "Task 1 image-only articleType": TASK1_DIR / "best_imageonly_articletype.pt",
    "Task 1 multi-input":            TASK1_DIR / "best_multiinput_task1.pt",
    "Task 1 metadata preprocessor":  TASK1_DIR / "metadata_preprocessor_task1.joblib",
    "Task 1 articleType encoder":    TASK1_DIR / "articletype_encoder_task1.joblib",
    "Task 2 multi-input":            TASK2_DIR / "best_multiinput_task2.pt",
    "Task 2 metadata preprocessor":  TASK2_DIR / "meta_preprocessor_task2.joblib",
    "Task 3 gender image-only":      TASK3_DIR / "gender_imageonly_smallcnn.pt",
    "Task 3 usage image-only":       TASK3_DIR / "usage_imageonly_smallcnn.pt",
    "Task 3 metadata one-hot":       TASK3_DIR / "ohe_metadata.joblib",
    "Task 3 gender encoder":         TASK3_DIR / "gender_encoder.joblib",
    "Task 3 usage encoder":          TASK3_DIR / "usage_encoder.joblib",
}

# Task 2 saves whichever image-only season model you trained; accept either.
TASK2_SEASON_CANDIDATES = [TASK2_DIR / "seresidual_cnn_image_only.pt",
                           TASK2_DIR / "improved_small_cnn_image_only.pt"]

missing = {k: v for k, v in REQUIRED_ARTIFACTS.items() if not v.exists()}
season_ckpt_path = next((p for p in TASK2_SEASON_CANDIDATES if p.exists()), None)

for name, path in REQUIRED_ARTIFACTS.items():
    print(f"[{'OK' if path.exists() else 'MISSING':>7}] {name:34s} {path}")
print(f"[{'OK' if season_ckpt_path else 'MISSING':>7}] {'Task 2 season image-only':34s} "
      f"{season_ckpt_path or TASK2_SEASON_CANDIDATES[0]}")

if missing or season_ckpt_path is None:
    raise FileNotFoundError(
        "Run Tasks 1-3 to completion first -- they produce the artifacts above. "
        f"Missing: {list(missing) + ([] if season_ckpt_path else ['Task 2 season image-only'])}")
print("\nAll required artifacts present.")

### 2. Architecture definitions

`torch.load` restores *weights*, not *structure*, so every architecture must be redeclared
here exactly as it was in the task notebooks. That duplication is a genuine fragility: edit
an encoder in a task notebook and this notebook will load stale weights into a mismatched
graph. Two defences:

- Every load below uses `strict=True` (PyTorch's default), so a shape or key mismatch raises
  rather than silently loading a partial state dict.
- Section 3 runs a forward pass on real images immediately after loading and checks the
  output shape against the expected class count.

If you keep iterating on architectures, move these classes into a shared `models.py` that all
four notebooks import — that removes the duplication entirely.

In [ ]:
import torch.nn.functional as F


# ── From Tasks 1 and 2: SE-residual building blocks ──────────────────────────
class SqueezeExcite(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        s = torch.sigmoid(self.fc2(F.silu(self.fc1(s))))
        return x * s[:, :, None, None]


class SEResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.se = SqueezeExcite(out_ch)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.shortcut = (nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
                         if (stride != 1 or in_ch != out_ch) else nn.Identity())

    def forward(self, x):
        out = F.silu(self.bn1(x))
        shortcut = self.shortcut(out if not isinstance(self.shortcut, nn.Identity) else x)
        out = self.conv1(out)
        out = self.conv2(F.silu(self.bn2(out)))
        out = self.se(self.drop(out))
        return out + shortcut


class SEResidualCNN(nn.Module):
    """Task 1's name for the encoder."""
    def __init__(self, out_dim=128, widths=(32, 64, 128, 256), drop=0.1):
        super().__init__()
        self.stem = nn.Conv2d(3, widths[0], 3, padding=1, bias=False)
        stages, in_ch = [], widths[0]
        for stage_idx, width in enumerate(widths):
            stride = 1 if stage_idx == 0 else 2
            stages.append(SEResidualBlock(in_ch, width, stride=stride, drop=drop))
            stages.append(SEResidualBlock(width, width, stride=1, drop=drop))
            in_ch = width
        self.stages = nn.Sequential(*stages)
        self.norm = nn.BatchNorm2d(in_ch)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim
        self.proj = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_ch, out_dim))

    def forward(self, x):
        x = self.stages(self.stem(x))
        x = self.pool(F.silu(self.norm(x))).flatten(1)
        return self.proj(x)


SEResidualEncoder = SEResidualCNN   # Task 2's name for the identical architecture


# ── From Task 1 ──────────────────────────────────────────────────────────────
class BasicCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim
        self.proj = nn.Linear(64, out_dim)

    def forward(self, x):
        return self.proj(self.pool(self.features(x)).flatten(1))


class VGGStyleCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding='same'), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding='same'), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding='same'), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding='same'), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, 3, padding='same'), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding='same'), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Conv2d(128, 256, 3, padding='same'), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding='same'), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Dropout2d(0.3),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim
        self.proj = nn.Linear(256, out_dim)

    def forward(self, x):
        return self.proj(self.pool(self.features(x)).flatten(1))


class ResNetStyleCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        from torchvision.models import resnet18
        backbone = resnet18(weights=None)          # from scratch, as in Task 1
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.out_dim = out_dim
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class Classifier(nn.Module):
    """Task 1's image-only wrapper."""
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.out_dim, n_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


class MetadataEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, out_dim), nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class MultiInputNetT1(nn.Module):
    """Task 1's fusion model. Attribute names must match the saved state dict."""
    def __init__(self, image_encoder, meta_dim, n_classes, meta_embedding_dim=64):
        super().__init__()
        self.image_encoder = image_encoder
        self.meta_encoder = MetadataEncoder(meta_dim, out_dim=meta_embedding_dim)
        self.classifier = nn.Sequential(
            nn.Linear(image_encoder.out_dim + meta_embedding_dim, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, n_classes),
        )

    def forward(self, img, meta):
        return self.classifier(torch.cat([self.image_encoder(img), self.meta_encoder(meta)], dim=1))


# ── From Tasks 2 and 3 ───────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels))

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act(out + residual)


class ImprovedSmallImageEncoder(nn.Module):
    def __init__(self, out_dim=128, dropout=0.2):
        super().__init__()
        self.out_dim = out_dim
        self.stage1 = ConvBlock(3, 32);   self.pool1 = nn.MaxPool2d(2)
        self.stage2 = ConvBlock(32, 64);  self.pool2 = nn.MaxPool2d(2)
        self.stage3 = ConvBlock(64, 128); self.pool3 = nn.MaxPool2d(2)
        self.stage4 = ConvBlock(128, 256)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, out_dim),
                                  nn.BatchNorm1d(out_dim), nn.SiLU())

    def forward(self, x):
        x = self.pool1(self.stage1(x))
        x = self.pool2(self.stage2(x))
        x = self.pool3(self.stage3(x))
        return self.proj(self.global_pool(self.stage4(x)).flatten(1))


class SmallImageEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.out_dim = out_dim
        self.proj = nn.Linear(128, out_dim)

    def forward(self, x):
        return self.proj(self.features(x).flatten(1))


class ResNet18Encoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        from torchvision.models import resnet18
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.out_dim = out_dim
        self.backbone = backbone
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class EfficientNetEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        from torchvision.models import efficientnet_b0
        backbone = efficientnet_b0(weights=None)
        self.out_dim = out_dim
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.proj = nn.Linear(1280, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class MultiInputNetT23(nn.Module):
    """Tasks 2 and 3's fusion model. Note Task 2's version carries a learned
    `meta_scale` parameter; Task 3's does not. Both are handled at load time."""
    def __init__(self, metadata_dim, n_classes, image_encoder, meta_scale=False):
        super().__init__()
        self.image = image_encoder
        if meta_scale:
            self.meta_scale = nn.Parameter(torch.tensor(1.0))
        else:
            self.meta_scale = None
        self.meta = nn.Sequential(
            nn.BatchNorm1d(metadata_dim), nn.Linear(metadata_dim, 128),
            nn.ReLU(), nn.Dropout(0.2))
        self.head = nn.Sequential(
            nn.Linear(image_encoder.out_dim + 128, 256), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(256, n_classes))

    def forward(self, image, metadata):
        meta_feats = self.meta(metadata)
        if self.meta_scale is not None:
            meta_feats = meta_feats * self.meta_scale
        return self.head(torch.cat([self.image(image), meta_feats], dim=1))


class ImageOnlyClassifier(nn.Module):
    """Tasks 2 and 3's image-only wrapper."""
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.out_dim, n_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


ENCODER_REGISTRY = {
    "SEResidualCNN": SEResidualCNN, "SEResidualEncoder": SEResidualEncoder,
    "BasicCNN": BasicCNN, "VGGStyleCNN": VGGStyleCNN, "ResNetStyleCNN": ResNetStyleCNN,
    "ImprovedSmallImageEncoder": ImprovedSmallImageEncoder,
    "SmallImageEncoder": SmallImageEncoder, "ResNet18Encoder": ResNet18Encoder,
    "EfficientNetEncoder": EfficientNetEncoder,
}
print(f"{len(ENCODER_REGISTRY)} encoder architectures registered.")

### 3. Datasets, loaders and the inference helper

One dataset class for inference on a bare list of ids — used identically for validation
images and test images, so the two paths cannot drift apart.

In [ ]:
class InferenceImageDataset(Dataset):
    """Images only, no labels. Returns (image_tensor, id) in the order given."""
    def __init__(self, ids, images_dir, transform):
        self.ids = list(ids)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]
        with Image.open(self.images_dir / f"{img_id}.jpg") as im:
            return self.transform(im), img_id


def make_inference_loader(ids, images_dir):
    return DataLoader(InferenceImageDataset(ids, images_dir, eval_transform),
                      batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))


@torch.no_grad()
def predict_image_only(model, loader):
    """Returns predicted class INDICES in loader order. shuffle=False everywhere,
    so position i of the output corresponds to position i of the id list."""
    model.eval()
    preds = []
    for imgs, _ in tqdm(loader, desc="  image-only inference", leave=False):
        preds.append(model(imgs.to(DEVICE)).argmax(1).cpu())
    return torch.cat(preds).numpy()


@torch.no_grad()
def predict_multi_input(model, loader, meta_matrix):
    """Same, for a fusion model. `meta_matrix` must be row-aligned with the loader's
    id list -- the assertion below is the guard against a silent misalignment, which
    would produce plausible-looking but meaningless predictions."""
    model.eval()
    assert len(meta_matrix) == len(loader.dataset), \
        f"metadata rows ({len(meta_matrix)}) != images ({len(loader.dataset)})"
    preds, cursor = [], 0
    for imgs, _ in tqdm(loader, desc="  multi-input inference", leave=False):
        bs = imgs.size(0)
        meta = torch.as_tensor(meta_matrix[cursor:cursor + bs], dtype=torch.float32).to(DEVICE)
        preds.append(model(imgs.to(DEVICE), meta).argmax(1).cpu())
        cursor += bs
    assert cursor == len(meta_matrix), "consumed fewer metadata rows than expected"
    return torch.cat(preds).numpy()


def load_state(model, path_or_ckpt, label):
    """strict=True load with a clear failure message naming the culprit."""
    ckpt = torch.load(path_or_ckpt, map_location=DEVICE) if not isinstance(path_or_ckpt, dict) else path_or_ckpt
    state = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    try:
        model.load_state_dict(state, strict=True)
    except RuntimeError as e:
        raise RuntimeError(
            f"{label}: saved weights do not match the architecture declared in Section 2. "
            f"This almost always means the encoder was edited in the task notebook after "
            f"the checkpoint was written -- retrain or sync the class definition.\n\n{e}")
    return model.to(DEVICE).eval()

### 4. Loading the Stage 1 attribute models

Each load is followed immediately by a forward pass on real validation images and a shape
check. A model that loads cleanly but produces the wrong number of logits would otherwise
only reveal itself much later, as a confusing indexing error.

In [ ]:
attribute_models = {}
val_probe_ids = val_data['id'].head(BATCH_SIZE).tolist()
probe_loader = make_inference_loader(val_probe_ids, IMAGES_TRAIN_DIR)


def register_attribute_model(name, model, encoder, n_expected):
    """Load-and-verify: run a real batch through and confirm the logit width."""
    with torch.no_grad():
        imgs, _ = next(iter(probe_loader))
        logits = model(imgs.to(DEVICE))
    assert logits.shape[1] == n_expected, \
        f"{name}: model outputs {logits.shape[1]} classes, encoder has {n_expected}"
    attribute_models[name] = (model, encoder)
    print(f"  [OK] {name:14s} {logits.shape[1]:3d} classes  ({type(model.encoder).__name__})")


print("Stage 1 attribute models:")

# --- articleType (Task 1) ---
ck = torch.load(TASK1_DIR / "best_imageonly_articletype.pt", map_location=DEVICE)
enc_at = joblib.load(TASK1_DIR / "articletype_encoder_task1.joblib")
m = Classifier(ENCODER_REGISTRY[ck["encoder_class"]](out_dim=ck["out_dim"]), ck["n_classes"])
register_attribute_model("articleType", load_state(m, ck, "Task 1 articleType"),
                         enc_at, len(enc_at.classes_))

# --- season (Task 2) ---
ck = torch.load(season_ckpt_path, map_location=DEVICE)
enc_se = encoders['season']
enc_cls = SEResidualEncoder if "seresidual" in season_ckpt_path.name else ImprovedSmallImageEncoder
m = ImageOnlyClassifier(enc_cls(out_dim=ck.get("out_dim", 128)), ck["n_classes"])
register_attribute_model("season", load_state(m, ck, "Task 2 season"), enc_se, len(enc_se.classes_))

# --- gender and usage (Task 3) ---
for attr, fname, enc_file in [("gender", "gender_imageonly_smallcnn.pt", "gender_encoder.joblib"),
                              ("usage", "usage_imageonly_smallcnn.pt", "usage_encoder.joblib")]:
    enc = joblib.load(TASK3_DIR / enc_file)
    m = ImageOnlyClassifier(ImprovedSmallImageEncoder(out_dim=128), len(enc.classes_))
    register_attribute_model(attr, load_state(m, TASK3_DIR / fname, f"Task 3 {attr}"),
                             enc, len(enc.classes_))

print(f"\n{len(attribute_models)} of 5 attribute models loaded "
      f"(baseColour is trained in Section 5).")

### 5. The missing model: `baseColour`

No task predicts `baseColour`, but Tasks 1, 2 and 3 all consume it — Task 1 flagged this as
outstanding work in its Step 14. It is trained here, using exactly the same machinery as
every other model: the shared split, the shared transforms, a `WeightedRandomSampler`, and a
from-scratch encoder.

`baseColour` has 46 categories in train with a long tail (the rarest has 2 rows), so the same
rare-class treatment used elsewhere applies: categories below a train-count threshold are
folded into `Other`. The threshold is set from **train counts only**, consistent with every
other threshold in this project.

In [ ]:
BASECOLOUR_MIN_COUNT = 50

vc_colour = train_data['baseColour'].value_counts()
print(f"baseColour categories in train: {len(vc_colour)}")
for thresh in [10, 25, 50, 100]:
    kept = (vc_colour >= thresh).sum()
    rows = vc_colour[vc_colour >= thresh].sum()
    print(f"  threshold={thresh:>4}: {kept:>2}/{len(vc_colour)} categories kept, "
          f"{rows} rows ({rows/len(train_data)*100:.1f}%)")

rare_colours = set(vc_colour[vc_colour < BASECOLOUR_MIN_COUNT].index)
for d in (train_data, val_data):
    d['baseColour_grouped'] = d['baseColour'].where(~d['baseColour'].isin(rare_colours), 'Other')

colour_encoder = LabelEncoder().fit(train_data['baseColour_grouped'])
N_COLOUR = len(colour_encoder.classes_)
train_data['baseColour_enc'] = colour_encoder.transform(train_data['baseColour_grouped'])

# Validation rows whose colour never appears in train can't be scored; none should exist
# after grouping, but check rather than assume.
val_known = val_data['baseColour_grouped'].isin(colour_encoder.classes_)
if (~val_known).sum():
    print(f"Dropping {(~val_known).sum()} val row(s) with an unseen baseColour")
    val_data = val_data[val_known].reset_index(drop=True)
val_data['baseColour_enc'] = colour_encoder.transform(val_data['baseColour_grouped'])

print(f"\nbaseColour classes after grouping: {N_COLOUR}")
print(train_data['baseColour_grouped'].value_counts())

In [ ]:
COLOUR_EPOCHS, COLOUR_PATIENCE = 30, 5

colour_train_ds = FashionImageDataset(train_data, IMAGES_TRAIN_DIR, 'baseColour_enc', train_transform)
colour_val_ds = FashionImageDataset(val_data, IMAGES_TRAIN_DIR, 'baseColour_enc', eval_transform)

colour_sampler = get_weighted_sampler(train_data, 'baseColour_enc')
colour_train_loader = DataLoader(colour_train_ds, batch_size=BATCH_SIZE, sampler=colour_sampler,
                                 num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))
colour_val_loader = DataLoader(colour_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))


def fit_simple(model, loader_tr, loader_va, name, epochs, patience, lr=3e-4, weight_decay=1e-4):
    """Same recipe as the task notebooks: AdamW, ReduceLROnPlateau, early stopping on
    validation loss, best-checkpoint restore."""
    criterion = nn.CrossEntropyLoss()
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=2)
    best_loss, best_state, best_f1, best_epoch, bad = None, None, None, None, 0
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

    for epoch in range(epochs):
        for phase, loader in (('train', loader_tr), ('val', loader_va)):
            training = phase == 'train'
            model.train() if training else model.eval()
            total, n, preds, actual = 0.0, 0, [], []
            with torch.set_grad_enabled(training):
                for imgs, labels in tqdm(loader, desc=f'{name} {epoch+1}/{epochs} {phase}', leave=False):
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                    if training:
                        optimiser.zero_grad(); loss.backward(); optimiser.step()
                    total += loss.item() * labels.size(0); n += labels.size(0)
                    preds.extend(logits.argmax(1).detach().cpu().numpy())
                    actual.extend(labels.cpu().numpy())
            epoch_loss, epoch_f1 = total / n, f1_score(actual, preds, average='macro', zero_division=0)
            history[f'{phase}_loss'].append(epoch_loss); history[f'{phase}_f1'].append(epoch_f1)

        val_loss, val_f1 = history['val_loss'][-1], history['val_f1'][-1]
        print(f'Epoch {epoch+1}/{epochs} - train_loss: {history["train_loss"][-1]:.4f} - '
              f'val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f}')
        scheduler.step(val_loss)
        if best_loss is None or val_loss < best_loss:
            best_loss, best_f1, best_epoch, bad = val_loss, val_f1, epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience:
                print(f'Early stopping at epoch {epoch+1} (best was {best_epoch})')
                break

    model.load_state_dict(best_state)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(name, fontweight='bold')
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].plot(history['train_f1'], label='Train'); axes[1].plot(history['val_f1'], label='Val')
    axes[1].set_title('Macro-F1'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout(); plt.show()
    print(f'>>> {name}: best epoch {best_epoch}, val_loss={best_loss:.4f}, val_f1={best_f1:.4f}')
    return model, best_f1


import copy
torch.manual_seed(RANDOM_STATE)
colour_model = ImageOnlyClassifier(SEResidualCNN(out_dim=128), N_COLOUR).to(DEVICE)
colour_model, colour_f1 = fit_simple(colour_model, colour_train_loader, colour_val_loader,
                                     'baseColour CNN (from scratch)', COLOUR_EPOCHS, COLOUR_PATIENCE)

torch.save({'model_state_dict': colour_model.state_dict(), 'n_classes': N_COLOUR,
            'out_dim': 128, 'encoder_class': 'SEResidualCNN',
            'val_macro_f1': float(colour_f1), 'pretrained': False,
            'min_count': BASECOLOUR_MIN_COUNT},
           PIPELINE_DIR / 'basecolour_cnn.pt')
joblib.dump(colour_encoder, PIPELINE_DIR / 'basecolour_encoder.joblib')
attribute_models['baseColour'] = (colour_model, colour_encoder)
print(f"\nSaved baseColour model. {len(attribute_models)} attribute models ready.")

### 6. Derived and constant metadata

`subCategory` and `masterCategory` need no model. In this dataset each `articleType` belongs
to exactly one `subCategory`, and each `subCategory` to exactly one `masterCategory` — the
three columns form a fixed taxonomy, not three independent labels. So once `articleType` is
predicted, the other two follow by lookup.

That is an assumption about the data, so it is **verified rather than trusted**: the cell
below checks that each child maps to exactly one parent, and prints any violation instead of
silently taking the most common parent.

`year` gets the training mode. Task 1's Step 14 already named this as the plan, and it is the
right call — `year` is a catalogue timestamp with no visual signature, so a CNN would be
predicting noise. A constant is honest about carrying no information.

In [ ]:
# --- verify the taxonomy really is a function -------------------------------
at_to_sub = train_data.groupby('articleType')['subCategory'].nunique()
sub_to_master = train_data.groupby('subCategory')['masterCategory'].nunique()

ambiguous_at = at_to_sub[at_to_sub > 1]
ambiguous_sub = sub_to_master[sub_to_master > 1]

print(f"articleType values mapping to >1 subCategory:   {len(ambiguous_at)}")
print(f"subCategory values mapping to >1 masterCategory: {len(ambiguous_sub)}")
if len(ambiguous_at):
    print("\nAmbiguous articleType -> subCategory:"); print(ambiguous_at)
if len(ambiguous_sub):
    print("\nAmbiguous subCategory -> masterCategory:"); print(ambiguous_sub)

# Use the most common parent where ambiguity exists, but only after showing it above --
# the point is that the fallback is visible, not hidden.
ARTICLETYPE_TO_SUB = (train_data.groupby('articleType')['subCategory']
                      .agg(lambda s: s.mode().iloc[0]).to_dict())
SUB_TO_MASTER = (train_data.groupby('subCategory')['masterCategory']
                 .agg(lambda s: s.mode().iloc[0]).to_dict())

# The articleType model predicts articleType_grouped, whose rare values ARE subCategory
# names (Section II.3 folds rare types into their subCategory). Those map to themselves.
for sub in train_data['subCategory'].unique():
    ARTICLETYPE_TO_SUB.setdefault(sub, sub)

YEAR_MODE = float(pd.to_numeric(train_data['year'], errors='coerce').mode().iloc[0])

print(f"\nTaxonomy: {len(ARTICLETYPE_TO_SUB)} articleType -> subCategory, "
      f"{len(SUB_TO_MASTER)} subCategory -> masterCategory")
print(f"year constant (train mode): {YEAR_MODE:.0f}")

# Every class the articleType model can emit must have a subCategory mapping, or the
# pipeline would produce NaN for some rows.
unmapped = [c for c in attribute_models['articleType'][1].classes_ if c not in ARTICLETYPE_TO_SUB]
assert not unmapped, f"articleType classes with no subCategory mapping: {unmapped}"
print("All articleType classes resolve to a subCategory.")

### 7. Stage 1 — the metadata generator

One function, used for both validation and test. Everything downstream depends on this
returning a frame whose rows are in the same order as the ids it was given, so the id column
is returned alongside and checked by the caller.

In [ ]:
METADATA_COLUMNS = ['gender', 'masterCategory', 'subCategory', 'articleType',
                    'baseColour', 'season', 'usage', 'year']


def generate_metadata(ids, images_dir, verbose=True):
    """Stage 1: predict every metadata column from images alone.

    Returns a DataFrame indexed 0..n-1, row-aligned with `ids`. Uses ONLY image-only
    models, so nothing here depends on any multi-input model -- that is what keeps the
    chain acyclic.
    """
    ids = [str(i) for i in ids]
    loader = make_inference_loader(ids, images_dir)
    out = pd.DataFrame({'id': ids})

    for attr in ['articleType', 'gender', 'season', 'usage', 'baseColour']:
        model, encoder = attribute_models[attr]
        if verbose:
            print(f"  predicting {attr} ...")
        codes = predict_image_only(model, loader)
        out[attr] = encoder.inverse_transform(codes)

    # Derived, not predicted.
    out['subCategory'] = out['articleType'].map(ARTICLETYPE_TO_SUB)
    out['masterCategory'] = out['subCategory'].map(SUB_TO_MASTER)
    out['year'] = YEAR_MODE

    assert len(out) == len(ids) and list(out['id']) == ids, "row alignment broken"
    assert out[METADATA_COLUMNS].notna().all().all(), \
        f"NaN in generated metadata:\n{out[METADATA_COLUMNS].isna().sum()}"
    return out


# Smoke test on a handful of validation images before committing to the full run.
smoke_ids = val_data['id'].head(8).tolist()
smoke = generate_metadata(smoke_ids, IMAGES_TRAIN_DIR, verbose=False)
print("Smoke test -- generated metadata for 8 validation images:\n")
print(smoke[['id'] + METADATA_COLUMNS].to_string(index=False))
print("\nTrue values for the same 8 rows:\n")
print(val_data.head(8)[['id', 'gender', 'masterCategory', 'subCategory', 'articleType',
                        'baseColour', 'season', 'usage', 'year']].to_string(index=False))

### 8. Validation A — does the chain actually help?

**This is the section that decides what gets submitted.**

The chain is run end-to-end on the validation split, using *predicted* metadata, and each
task's multi-input model is scored three ways:

| Score | Meaning |
|---|---|
| **Oracle** | multi-input model fed the *true* metadata — the number the task notebooks report |
| **Chained** | multi-input model fed *predicted* metadata — what a real submission would score |
| **Image-only** | no metadata at all — the deployable alternative |

The gap between oracle and chained is the cost of the train/serve skew. If chained falls
below image-only, the chain is a net loss for that task and the image-only model should be
submitted instead. Section 9 applies that rule automatically.

In [ ]:
# Stage 1 on the full validation split -- predicted metadata, never true values.
print("Generating predicted metadata for the validation split...")
val_ids = val_data['id'].tolist()
val_meta_pred = generate_metadata(val_ids, IMAGES_TRAIN_DIR)

# How good is each generated column? This is the leading indicator: if an attribute
# model is weak, the multi-input models that consume it will be fed noise.
print("\nStage 1 attribute quality on validation (accuracy of each generated column):")
attr_quality = {}
truth = {'articleType': val_data['articleType_grouped'], 'gender': val_data['gender'],
         'season': val_data['season'], 'usage': val_data['usage_grouped'],
         'baseColour': val_data['baseColour_grouped']}
for attr, true_series in truth.items():
    mask = true_series.notna().to_numpy()
    acc = accuracy_score(true_series[mask], val_meta_pred.loc[mask, attr])
    f1m = f1_score(true_series[mask], val_meta_pred.loc[mask, attr],
                   average='macro', zero_division=0)
    attr_quality[attr] = {'accuracy': acc, 'macro_f1': f1m, 'n': int(mask.sum())}
    print(f"  {attr:12s} accuracy={acc:.3f}  macro-F1={f1m:.3f}  (n={mask.sum()})")

attr_quality_df = pd.DataFrame(attr_quality).T.round(4)
display(attr_quality_df)

In [ ]:
# ── Load the Stage 2 multi-input models ─────────────────────────────────────
multi_models = {}

# Task 1
info1 = json.load(open(TASK1_DIR / "best_multiinput_task1_info.json"))
enc_cls = ENCODER_REGISTRY[info1["image_encoder"]]
m1 = MultiInputNetT1(enc_cls(), meta_dim=info1["metadata_dim"], n_classes=info1["n_classes"])
multi_models['articleType'] = load_state(m1, TASK1_DIR / "best_multiinput_task1.pt", "Task 1 multi-input")
prep1 = joblib.load(TASK1_DIR / "metadata_preprocessor_task1.joblib")

# Task 2
ck2 = torch.load(TASK2_DIR / "best_multiinput_task2.pt", map_location=DEVICE)
m2 = MultiInputNetT23(ck2["metadata_dim"], ck2["n_classes"],
                      ENCODER_REGISTRY[ck2["image_encoder"]](out_dim=ck2["out_dim"]),
                      meta_scale=True)
multi_models['season'] = load_state(m2, ck2, "Task 2 multi-input")
prep2 = joblib.load(TASK2_DIR / "meta_preprocessor_task2.joblib")

# Task 3
ohe3 = joblib.load(TASK3_DIR / "ohe_metadata.joblib")
enc_g3 = joblib.load(TASK3_DIR / "gender_encoder.joblib")
enc_u3 = joblib.load(TASK3_DIR / "usage_encoder.joblib")
meta_dim3 = len(ohe3.get_feature_names_out())
for attr, fname, enc3 in [('gender', 'gender_multiinput_smallcnn.pt', enc_g3),
                          ('usage', 'usage_multiinput_smallcnn.pt', enc_u3)]:
    m3 = MultiInputNetT23(meta_dim3, len(enc3.classes_),
                          ImprovedSmallImageEncoder(out_dim=128), meta_scale=False)
    multi_models[attr] = load_state(m3, TASK3_DIR / fname, f"Task 3 {attr} multi-input")

print(f"Loaded {len(multi_models)} multi-input models: {list(multi_models)}")


# ── Build each task's metadata matrix with its OWN fitted preprocessor ───────
# Each task fit a different transformer on a different column set. Using the wrong one
# would produce a matrix of the wrong width (caught) or the right width with columns in
# the wrong order (NOT caught by any shape check) -- hence one builder per task.
T1_CAT, T1_NUM = ['gender', 'baseColour', 'season', 'usage'], ['year']
T2_COLS = ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'year', 'usage']
T2_CAT = [c for c in T2_COLS if c != 'year']
T3_COLS = ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'year', 'season']


def build_matrix_task1(meta_df):
    frame = meta_df[T1_CAT + T1_NUM].copy()
    for c in T1_CAT:
        frame[c] = frame[c].astype(object)
    frame['year'] = pd.to_numeric(frame['year'], errors='coerce').astype(float)
    X = prep1.transform(frame)
    return X.toarray().astype('float32') if hasattr(X, 'toarray') else X.astype('float32')


def build_matrix_task2(meta_df):
    frame = meta_df[T2_COLS].copy()
    for c in T2_CAT:
        frame[c] = frame[c].astype(object)
    frame['year'] = pd.to_numeric(frame['year'], errors='coerce')
    X = prep2.transform(frame)
    return X.toarray().astype('float32') if hasattr(X, 'toarray') else X.astype('float32')


def build_matrix_task3(meta_df):
    frame = meta_df[T3_COLS].copy()
    for c in T3_COLS:
        if c != 'year':
            frame[c] = frame[c].astype(object)
    frame['year'] = pd.to_numeric(frame['year'], errors='coerce')
    return ohe3.transform(frame).astype('float32')


MATRIX_BUILDERS = {'articleType': build_matrix_task1, 'season': build_matrix_task2,
                   'gender': build_matrix_task3, 'usage': build_matrix_task3}
EXPECTED_WIDTH = {'articleType': info1["metadata_dim"], 'season': ck2["metadata_dim"],
                  'gender': meta_dim3, 'usage': meta_dim3}

for task, builder in MATRIX_BUILDERS.items():
    X = builder(val_meta_pred)
    assert X.shape == (len(val_meta_pred), EXPECTED_WIDTH[task]), \
        (f"{task}: metadata matrix is {X.shape}, model expects "
         f"({len(val_meta_pred)}, {EXPECTED_WIDTH[task]}). The saved preprocessor and the "
         f"saved model came from different runs -- retrain or re-export both together.")
    print(f"  [OK] {task:12s} metadata matrix {X.shape}")

In [ ]:
# ── Score all three ways ─────────────────────────────────────────────────────
val_loader_pipe = make_inference_loader(val_ids, IMAGES_TRAIN_DIR)

TASK_TRUTH = {'articleType': val_data['articleType_grouped'], 'season': val_data['season'],
              'gender': val_data['gender'], 'usage': val_data['usage_grouped']}
TASK_ENCODER = {'articleType': attribute_models['articleType'][1], 'season': encoders['season'],
                'gender': enc_g3, 'usage': enc_u3}

# The true-metadata frame, built from val_data with the SAME column names Stage 1 emits.
val_meta_true = pd.DataFrame({
    'id': val_data['id'].astype(str), 'gender': val_data['gender'],
    'masterCategory': val_data['masterCategory'], 'subCategory': val_data['subCategory'],
    'articleType': val_data['articleType_grouped'], 'baseColour': val_data['baseColour_grouped'],
    'season': val_data['season'], 'usage': val_data['usage_grouped'],
    'year': pd.to_numeric(val_data['year'], errors='coerce'),
}).reset_index(drop=True)
# Oracle metadata still has genuine NaNs (season/usage are missing on some rows); the
# fitted imputers inside each preprocessor handle those, exactly as during training.

comparison_rows = []
for task in ['articleType', 'season', 'gender', 'usage']:
    truth_series = TASK_TRUTH[task]
    mask = truth_series.notna().to_numpy()
    y_true = TASK_ENCODER[task].transform(truth_series[mask])

    print(f"\n=== {task} ===")
    chained = predict_multi_input(multi_models[task], val_loader_pipe,
                                  MATRIX_BUILDERS[task](val_meta_pred))[mask]
    oracle = predict_multi_input(multi_models[task], val_loader_pipe,
                                 MATRIX_BUILDERS[task](val_meta_true))[mask]
    img_only = predict_image_only(attribute_models[task][0], val_loader_pipe)[mask]

    row = {
        'task': task,
        'oracle_metadata': f1_score(y_true, oracle, average='macro', zero_division=0),
        'chained_metadata': f1_score(y_true, chained, average='macro', zero_division=0),
        'image_only': f1_score(y_true, img_only, average='macro', zero_division=0),
    }
    row['skew_cost'] = row['oracle_metadata'] - row['chained_metadata']
    row['chain_gain_vs_image_only'] = row['chained_metadata'] - row['image_only']
    row['use_for_submission'] = 'multi-input (chained)' if row['chain_gain_vs_image_only'] > 0 else 'image-only'
    comparison_rows.append(row)
    print(f"  oracle={row['oracle_metadata']:.4f}  chained={row['chained_metadata']:.4f}  "
          f"image-only={row['image_only']:.4f}  -> {row['use_for_submission']}")

chain_comparison = pd.DataFrame(comparison_rows).set_index('task').round(4)
display(chain_comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

x = np.arange(len(chain_comparison)); width = 0.27
for offset, col, colour, label in [(-width, 'oracle_metadata', '#A8C8EC', 'Oracle metadata'),
                                   (0, 'chained_metadata', '#4C72B0', 'Chained (predicted)'),
                                   (width, 'image_only', '#DD8452', 'Image-only')]:
    bars = axes[0].bar(x + offset, chain_comparison[col], width, label=label, color=colour)
    axes[0].bar_label(bars, fmt='%.3f', fontsize=8, padding=2)
axes[0].set_xticks(x); axes[0].set_xticklabels(chain_comparison.index)
axes[0].set_ylabel('Macro-F1'); axes[0].set_ylim(0, 1); axes[0].legend()
axes[0].set_title('Does predicted metadata survive the chain?')
axes[0].grid(axis='y', alpha=.25)

gains = chain_comparison['chain_gain_vs_image_only']
bars = axes[1].bar(x, gains, color=['#55A868' if g > 0 else '#C44E52' for g in gains])
axes[1].bar_label(bars, fmt='%+.3f', fontsize=9, padding=2)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xticks(x); axes[1].set_xticklabels(chain_comparison.index)
axes[1].set_ylabel('Macro-F1 gain'); axes[1].set_title('Chained minus image-only\n(green = chain helps)')
axes[1].grid(axis='y', alpha=.25)

plt.tight_layout(); plt.show()

print("Reading this chart:")
print("  oracle - chained  = the cost of feeding predicted instead of true metadata")
print("  chained - image-only = whether the chain was worth building at all")

### 9. Validation B — structural checks on the test metadata

Validation A asks whether the pipeline is *useful*. This section asks whether it *runs
correctly* on the real test set — a different question, and one that has to pass regardless
of which model wins.

Nine checks, each of which catches a failure that would otherwise produce a plausible-looking
but wrong submission file.

In [ ]:
test_df = pd.read_csv(TEST_PRED_CSV)
test_df['id'] = test_df['id'].astype(str).str.strip()
TEST_TEMPLATE_COLUMNS = list(test_df.columns)
print(f"Test template: {test_df.shape[0]} rows, columns {TEST_TEMPLATE_COLUMNS}")

checks = []


def check(name, passed, detail=""):
    checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))
    return passed


# 1. Every test id has an image file.
missing_imgs = [i for i in test_df['id'] if not (IMAGES_TEST_DIR / f"{i}.jpg").exists()]
check("every test id has an image file", not missing_imgs,
      f"{len(missing_imgs)} missing" if missing_imgs else f"{len(test_df)} images found")

# 2. Test ids are unique.
check("test ids are unique", test_df['id'].is_unique,
      f"{test_df['id'].duplicated().sum()} duplicates")

# 3. Test ids don't overlap the training set (they shouldn't -- separate folders).
overlap = set(test_df['id']) & set(df['id'].astype(str))
check("no test id appears in the training data", not overlap, f"{len(overlap)} overlapping")

# 4. Every test image opens and is the expected size.
bad_imgs, sizes = [], []
for img_id in test_df['id'].head(500):
    try:
        with Image.open(IMAGES_TEST_DIR / f"{img_id}.jpg") as im:
            sizes.append(im.size)
    except Exception as e:
        bad_imgs.append((img_id, str(e)))
check("test images open cleanly (500-image sample)", not bad_imgs,
      f"most common size {pd.Series(sizes).value_counts().index[0]}" if sizes else "")

# 5. A single batch survives the full transform.
probe = make_inference_loader(test_df['id'].head(BATCH_SIZE).tolist(), IMAGES_TEST_DIR)
probe_imgs, probe_ids = next(iter(probe))
check("test images pass through eval_transform",
      probe_imgs.shape[1:] == (3, IMG_HEIGHT, IMG_WIDTH),
      f"batch tensor {tuple(probe_imgs.shape)}")

# 6. Loader preserves id order (the assumption every alignment depends on).
check("DataLoader preserves id order",
      list(probe_ids) == test_df['id'].head(BATCH_SIZE).tolist())

In [ ]:
# ── Stage 1 on the real test set ─────────────────────────────────────────────
print("Generating metadata for the test set...")
test_meta = generate_metadata(test_df['id'].tolist(), IMAGES_TEST_DIR)

# 7. No missing values anywhere in the generated metadata.
check("generated test metadata has no NaN", test_meta[METADATA_COLUMNS].notna().all().all(),
      f"{int(test_meta[METADATA_COLUMNS].isna().sum().sum())} NaN cells")

# 8. Every generated category was seen during training -- an unseen value would be
#    silently dropped to all-zeros by handle_unknown='ignore' and quietly degrade results.
unseen_report = {}
for col in ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'usage']:
    train_vals = set(train_data[col].dropna().astype(str)) if col in train_data.columns else set()
    if col == 'articleType':
        train_vals = set(train_data['articleType_grouped'].dropna().astype(str))
    if col == 'usage':
        train_vals = set(train_data['usage_grouped'].dropna().astype(str))
    if col == 'baseColour':
        train_vals = set(train_data['baseColour_grouped'].dropna().astype(str))
    unseen = set(test_meta[col].astype(str)) - train_vals
    unseen_report[col] = unseen
check("no generated value is unseen in training",
      not any(unseen_report.values()),
      "; ".join(f"{k}: {v}" for k, v in unseen_report.items() if v) or "all values known")

# 9. Each task's metadata matrix has the exact width its model expects.
matrix_ok = True
for task, builder in MATRIX_BUILDERS.items():
    X = builder(test_meta)
    ok = X.shape == (len(test_meta), EXPECTED_WIDTH[task])
    matrix_ok &= ok
    print(f"      {task:12s} matrix {X.shape}, expected "
          f"({len(test_meta)}, {EXPECTED_WIDTH[task]})")
check("metadata matrices match the saved models' input widths", matrix_ok)

validation_report = pd.DataFrame(checks)
display(validation_report)

n_failed = (validation_report['result'] == 'FAIL').sum()
if n_failed:
    raise AssertionError(f"{n_failed} structural check(s) failed -- fix before submitting. "
                         "See the table above.")
print("\nAll structural checks passed.")

print("\nGenerated test metadata -- distribution of each column:")
for col in ['gender', 'season', 'usage', 'baseColour']:
    print(f"\n{col}:"); print(test_meta[col].value_counts().head(8))

### 10. Final predictions

Stage 2. For each of the four target columns, the model chosen in Section 8 runs on the test
images — the chained multi-input model where it beat image-only on validation, the image-only
model where it didn't. That decision is read from `chain_comparison`, not hard-coded, so it
follows the evidence rather than the assumption.

In [ ]:
test_loader = make_inference_loader(test_df['id'].tolist(), IMAGES_TEST_DIR)
submission = test_df[['id']].copy()
decision_log = []

for task in ['articleType', 'season', 'gender', 'usage']:
    use_chain = chain_comparison.loc[task, 'use_for_submission'] == 'multi-input (chained)'
    encoder = TASK_ENCODER[task]

    print(f"\n=== {task}: {'chained multi-input' if use_chain else 'image-only'} ===")
    if use_chain:
        codes = predict_multi_input(multi_models[task], test_loader, MATRIX_BUILDERS[task](test_meta))
        expected_f1 = chain_comparison.loc[task, 'chained_metadata']
    else:
        codes = predict_image_only(attribute_models[task][0], test_loader)
        expected_f1 = chain_comparison.loc[task, 'image_only']

    submission[task] = encoder.inverse_transform(codes)
    decision_log.append({'task': task,
                         'model': 'multi-input (chained)' if use_chain else 'image-only',
                         'expected_macro_f1': round(float(expected_f1), 4)})

decision_df = pd.DataFrame(decision_log).set_index('task')
display(decision_df)

In [ ]:
# ── Final submission checks ──────────────────────────────────────────────────
submission = submission[TEST_TEMPLATE_COLUMNS]   # exact template column order

final_checks = []


def final_check(name, passed, detail=""):
    final_checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))


final_check("column names and order match the template",
            list(submission.columns) == TEST_TEMPLATE_COLUMNS, str(list(submission.columns)))
final_check("row count matches the template",
            len(submission) == len(test_df), f"{len(submission)} vs {len(test_df)}")
final_check("id order matches the template",
            list(submission['id']) == list(test_df['id']))
final_check("no empty predictions",
            submission[['gender', 'articleType', 'season', 'usage']].notna().all().all(),
            f"{int(submission.isna().sum().sum())} NaN cells")
for col, enc in [('gender', enc_g3), ('season', encoders['season']),
                 ('usage', enc_u3), ('articleType', TASK_ENCODER['articleType'])]:
    bad = set(submission[col]) - set(enc.classes_)
    final_check(f"every predicted {col} is a valid class label", not bad, str(bad) if bad else "")

# A model that collapsed to one class would still pass every check above.
for col in ['gender', 'articleType', 'season', 'usage']:
    n_unique = submission[col].nunique()
    top_share = submission[col].value_counts(normalize=True).iloc[0]
    final_check(f"{col} predictions are not degenerate", n_unique > 1 and top_share < 0.95,
                f"{n_unique} distinct values, most common {top_share:.1%}")

display(pd.DataFrame(final_checks))
if (pd.DataFrame(final_checks)['result'] == 'FAIL').any():
    raise AssertionError("Submission checks failed -- do not submit this file.")

SUBMISSION_PATH = PIPELINE_DIR / 'styles_prediction_filled.csv'
submission.to_csv(SUBMISSION_PATH, index=False)
test_meta.to_csv(PIPELINE_DIR / 'test_generated_metadata.csv', index=False)
chain_comparison.to_csv(PIPELINE_DIR / 'chain_validation_report.csv')

# Reload check: what's on disk is what we think it is.
reloaded = pd.read_csv(SUBMISSION_PATH)
reloaded['id'] = reloaded['id'].astype(str)
assert list(reloaded.columns) == TEST_TEMPLATE_COLUMNS and len(reloaded) == len(test_df)
assert reloaded[['gender', 'articleType', 'season', 'usage']].notna().all().all()

print(f"\nWrote {len(submission)} predictions to {SUBMISSION_PATH}")
print(f"Wrote generated metadata to {PIPELINE_DIR / 'test_generated_metadata.csv'}")
print(f"Wrote validation report to {PIPELINE_DIR / 'chain_validation_report.csv'}")
print("\nFirst rows of the submission:")
print(submission.head(10).to_string(index=False))

### 11. What to write in the report

**The pipeline's own numbers.** `chain_comparison` (Section 8) is the headline table. Quote
all three columns per task, not just the best one — the oracle-vs-chained gap is the
interesting finding, and it's the number that tells a marker you understood the difference
between a validation score and a deployable one.

**The one thing not to claim.** Task 1's 0.750 was measured with true metadata. Unless
`chained_metadata` comes out close to it, that number is not the expected test performance
and should never be quoted as such. The `skew_cost` column is exactly this gap, measured.

**If the chain lost.** That is a legitimate, reportable result, not a failed experiment.
It says the metadata's value lay in its *accuracy*, and predicted metadata isn't accurate
enough to carry it — which is precisely what Task 1's Step 11f predicted when it found
`gender` was the dominant contributor. Section 9's decision rule then submits the image-only
models, which is the right call.

**The known weakness, if you want to go further.** The multi-input models were trained on
clean metadata and served noisy metadata. The principled fix is to retrain them on
*cross-fitted predicted* metadata: split the training set into k folds, train the attribute
models on k−1 folds, predict the held-out fold, and assemble a training set whose metadata
has the same error distribution as the test set will. That removes the skew instead of
merely measuring it. It costs k times the attribute-model training and is out of scope here,
but naming it shows you know why the gap exists rather than just that it does.